# FoldPipe Quickstart

This notebook is the shortest path to the public API. It first runs a completely local example, so you can understand the loader without credentials or a remote dataset. The final cell shows how to swap in a revision-pinned Hugging Face source.

## 1. Install the released package

Install the version used by this tutorial from PyPI. Pinning the version keeps the notebook reproducible.

In [ ]:
%pip install -q foldpipe==0.3.1

## 2. Run a local, credential-free stream

`SyntheticLatencySource` produces four small tensor shards and waits briefly before returning each one. FoldPipe retrieves the next shard in a background thread while your loop consumes the current shard.

In [ ]:
import time
from foldpipe import AsyncFoldPipeLoader
from foldpipe.sources import SyntheticLatencySource

source = SyntheticLatencySource(
    num_chunks=4,
    latency_ms=150,
    chunk_size=1_024,
)
loader = AsyncFoldPipeLoader(source=source, batch_size=256)

started = time.perf_counter()
rows = 0
for batch_index, batch in enumerate(loader):
    rows += len(batch)
    if batch_index == 0:
        print("first batch shape:", tuple(batch.shape))

print(f"streamed {rows:,} rows in {time.perf_counter() - started:.2f} s")

## 3. Point the same loader at Hugging Face

Set `RUN_REMOTE = True` only after replacing the repository and revision placeholders. Keep tokens in `HF_TOKEN`; never paste them into a notebook. FoldPipe deserializes native `.pt` files, so use only shards you trust.

In [ ]:
import os
from foldpipe.sources import HuggingFaceSource

RUN_REMOTE = False
if RUN_REMOTE:
    remote_source = HuggingFaceSource(
        repo_id="YOUR_ACCOUNT/YOUR_DATASET",
        folder_path="",
        revision="PIN_A_COMMIT_SHA",
        token=os.environ.get("HF_TOKEN"),
    )
    remote_loader = AsyncFoldPipeLoader(remote_source, batch_size=128)
    first_remote_batch = next(iter(remote_loader))
    print(type(first_remote_batch), getattr(first_remote_batch, "shape", None))
else:
    print("Remote example is disabled; replace the placeholders before enabling it.")

## What FoldPipe guarantees

The number of live shard payloads is bounded independently of the total dataset size. The current shard and a completed prefetched shard can coexist, so an individually oversized shard can still exhaust memory. Prefetch creates overlap; it does not guarantee a speedup on every network or workload.

Next: open `Prion_Case_Study.ipynb` for an end-to-end educational example using the bundled 1QLX structure.